# 02 — Zero-shot baseline: Qwen2.5-0.5B-Instruct

Day 5 goal: establish a floor score and do a live check that the Day 4 eval
harness (`src/eval_metrics.py`) produces sane numbers against real model
output, not just the ref-vs-ref / ref-vs-shuffled synthetic sanity check.

Model: `Qwen/Qwen2.5-0.5B-Instruct` — chosen for this *local* smoke test
specifically because this machine is CPU-only with ~7.5GB RAM, so a small
model was needed to fit comfortably and generate in a reasonable time. The
actual Day 6 candidate shortlist will consider larger (up to 3B) models,
run on Kaggle's GPU.

Ran on: Aug 15, 2026 (local CPU env).

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from eval_metrics import evaluate

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
N = 15
SEED = 42

SYSTEM_PROMPT = (
    "আপনি একজন অভিজ্ঞ চিকিৎসক। একজন রোগী তার শারীরিক সমস্যা সম্পর্কে "
    "আপনাকে প্রশ্ন করেছেন। রোগীর প্রশ্নের উত্তর বাংলা ভাষায়, সহানুভূতিশীলভাবে "
    "এবং চিকিৎসাগতভাবে যথাযথভাবে দিন।"
)


## Load model, verify parameter count (must be <=3B)

In [1]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Param count: {n_params:,} ({n_params/1e9:.3f}B) -- under 3B cap: {n_params < 3_000_000_000}")


Param count: 494,032,768 (0.494B) -- under 3B cap: True


## Zero-shot generation on 15 val examples

In [1]:
val = pd.read_json("../data/processed/val.jsonl", lines=True).sample(
    n=N, random_state=SEED
).reset_index(drop=True)

preds = []
for i, row in val.iterrows():
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["input"]},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=150, do_sample=False, num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    preds.append(gen_text)
    print(f"[{i+1}/{N}] generated {len(gen_text)} chars")


[1/15] generated 147 chars
[2/15] generated 142 chars
[3/15] generated 35 chars
[4/15] generated 150 chars
[5/15] generated 136 chars
[6/15] generated 144 chars
[7/15] generated 145 chars
[8/15] generated 22 chars
[9/15] generated 142 chars
[10/15] generated 136 chars
[11/15] generated 140 chars
[12/15] generated 150 chars
[13/15] generated 139 chars
[14/15] generated 147 chars
[15/15] generated 149 chars


## Score with the Day 4 harness

In [1]:
refs = val["output"].tolist()
result = evaluate(preds, refs, verbose=False)

print(f"mean BERTScore F1: {result['mean_bertscore_f1']:.4f}")
print(f"mean Token F1:     {result['mean_token_f1']:.4f}")
print(f"mean ROUGE-L F1:   {result['mean_rouge_l_f1']:.4f}")
print(f"mean composite:    {result['mean_composite']:.4f}")


mean BERTScore F1: 0.6560
mean Token F1:     0.2625
mean ROUGE-L F1:   0.1710
mean composite:    0.4409


## Results

| Metric | ref-vs-ref (Day 4) | ref-vs-mismatched (Day 4) | **Qwen2.5-0.5B zero-shot (real model)** |
|---|---|---|---|
| BERTScore F1 | 1.0000 | 0.6854 | **0.6560** |
| Token F1 | 1.0000 | 0.5556 | **0.2625** |
| ROUGE-L F1 | 1.0000 | 0.2693 | **0.1710** |
| Composite | 1.0000 | 0.5632 | **0.4409** |

**Harness sanity confirmed on real model output**: the zero-shot composite
(0.44) sits meaningfully *below* even the ref-vs-mismatched synthetic
baseline (0.56). That's the expected direction — mismatched *real*
references still share fluent doctor-style Bengali and domain vocabulary
with each other, while a weak model's broken/repetitive output shares much
less. Token F1 and ROUGE-L (which require actual word/sequence overlap) are
the metrics that drop the hardest and are doing the real discriminating
work here; BERTScore's high floor (discussed in Day 4 notes) means it moves
much less.

## Sample generations

In [1]:
for i in range(3):
    print(f"--- Example {i+1} (composite={result['composite'][i]:.3f}) ---")
    print("INPUT:", val.iloc[i]["input"][:200])
    print("PRED: ", preds[i][:300])
    print("REF:  ", refs[i][:300])
    print()


--- Example 1 (composite=0.431) ---
INPUT: আমি ২০১২ সালের ফেব্রুয়ারিতে চোয়ালের অস্ত্রোপচার করেছিলাম। আমি এখন ক্রমাগত মাথাব্যথা, স্পন্দন এবং আমার কান ও নাকসহ মুখের চারপাশে অদ্ভুত অনুভূতি অনুভব করছি...
PRED:  এই সমস্যা আপনাকে প্রশ্ন করেছেন এবং এটি ছিল কোনও ক্ষেত্রে আপনাকে প্রশ্ন করেছেন। এটি আপনাকে কোনও সমস্যা করেছেন না। আপনি একটি অভিজ্ঞ চিকিৎসক এবং আপনা... (incoherent, repetitive loop)
REF:   হেলো হলি, লক্ষণগুলো গুরুতর, এবং ব্রেইন ইভেন্ট বা মস্তিষ্কের সমস্যা বাদ দেওয়ার জন্য আপনাকে সঠিকভাবেই সিটি/এমআরআই-এর জন্য স্ট্রোক ক্লিনিকে পাঠানো হয়েছে...

--- Example 3 (composite=0.366) ---
INPUT: হেলো, আমি ভারত (অরুণাচল প্রদেশ) থেকে বলছি...আমার আইএনআর (INR) পরীক্ষা করিয়েছি এবং ফলাফল ৬.২৫ এসেছে...
PRED:  আপনি আমার কাছে কিছু পরামর্শ চাইছেন? (just echoes the question back — non-answer)
REF:   প্রিয় কানো লেন্ড, আইএনআর (INR) ডেটা দুটি ক্ষেত্রে বৃদ্ধি পেতে পারে... (real, specific medical explanation)


## Takeaways for Day 6 (model shortlist)

1. **0.5B zero-shot is not viable** — outputs are frequently incoherent
   (repetitive loops) or non-answers (echoing the question). This is
   expected for a model this small without fine-tuning; it establishes the
   floor, not a candidate.
2. **Fine-tuning is essential**, not optional — the rulebook already
   permits/expects this, and this baseline confirms zero-shot prompting
   alone won't get anywhere near a competitive score at any model size in
   this budget.
3. For the Day 6 shortlist, prioritize models with **stronger native
   Bengali/Indic-language support** in the 1-3B range (e.g. Qwen2.5-1.5B/3B,
   Gemma-2-2b, or Bengali-specific checkpoints) over raw model size —
   generic multilingual coverage at 0.5B clearly isn't enough.
4. Per-example predictions saved to `experiments/day5_zeroshot_qwen05b.csv`
   for reference.